# Volatility surfaces

A volatility curve has one number per date. A surface has a number per date and per strike
coordinate. `VolatilitySurfaceDelta` uses put delta in percent: 50 is at the money, 10 and 90
are the wings. `VolatilitySurfaceMoneyness` uses `(strike - forward) / forward`. A
calendar-month moneyness surface reads the column for the contract month.

Each row of the volatility grid is one strike coordinate, and each column is an observation
date. A coordinate outside the grid reads the nearest wing. `extrapolate` extends the dates
only.


In [ ]:
import logging
import math
from datetime import date

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import matplotlib.pyplot as plt
import sjd

sjd.set_log_level("INFO")

anchor = date(2026, 9, 16)
pillars = [anchor, date(2026, 12, 16), date(2027, 9, 16)]
deltas = [10.0, 25.0, 50.0, 75.0, 90.0]
smile = [0.18, 0.15, 0.12, 0.15, 0.18]
grid = [
    [wing, wing + 0.01, wing + 0.02] for wing in smile
]
delta_surface = sjd.VolatilitySurfaceDelta(
    anchor, pillars, deltas, grid, sjd.SurfaceInterpolator.BILINEAR
)
assert delta_surface.validate().ok

forward = 100.0
horizon = pillars[-1]
strike_10 = delta_surface.strike_from_delta(horizon, 10.0, forward)
strike_50 = delta_surface.strike_from_delta(horizon, 50.0, forward)
strike_90 = delta_surface.strike_from_delta(horizon, 90.0, forward)
assert strike_10 < strike_50 < strike_90
assert abs(strike_50 - forward) / forward < 0.02
print(f"10 delta {strike_10:.4f}")
print(f"50 delta {strike_50:.4f}")
print(f"90 delta {strike_90:.4f}")


## Moneyness

Zero moneyness is the forward. A strike beyond the last row reads that row's volatility, on
both wings. With extrapolation left on, a date after the last pillar still returns a
volatility. On this surface the columns are flat, so the later date matches the last pillar.


In [ ]:
moneyness = [-0.10, 0.0, 0.10]
flat_grid = [[0.30, 0.30], [0.20, 0.20], [0.35, 0.35]]
moneyness_surface = sjd.VolatilitySurfaceMoneyness(
    anchor,
    [anchor, horizon],
    moneyness,
    flat_grid,
    sjd.SurfaceInterpolator.BILINEAR,
)
assert moneyness_surface.validate().ok
assert math.isclose(moneyness_surface.get_volatility(horizon, 100.0, 100.0), 0.20)
assert math.isclose(moneyness_surface.get_volatility(horizon, 70.0, 100.0), 0.30)
assert math.isclose(moneyness_surface.get_volatility(horizon, 140.0, 100.0), 0.35)
later = date(2028, 1, 15)
assert math.isclose(
    moneyness_surface.get_volatility(later, 100.0, 100.0),
    moneyness_surface.get_volatility(horizon, 100.0, 100.0),
)

monthly = sjd.VolatilitySurfaceMoneynessCMA(
    anchor,
    [date(2026, 10, 1), date(2026, 12, 1)],
    moneyness,
    [[0.30, 0.28], [0.20, 0.22], [0.35, 0.33]],
    sjd.SurfaceInterpolator.BILINEAR,
)
assert monthly.validate().ok
assert math.isclose(
    monthly.get_volatility(date(2026, 10, 5), 100.0, 100.0),
    monthly.get_volatility(date(2026, 10, 28), 100.0, 100.0),
)
assert not math.isclose(
    monthly.get_volatility(date(2026, 10, 5), 100.0, 100.0),
    monthly.get_volatility(date(2026, 12, 2), 100.0, 100.0),
)


## Bilinear and bicubic

The same delta grid interpolated both ways. The smile is read by converting each strike to a
delta and interpolating the surface there.


In [ ]:
bicubic = sjd.VolatilitySurfaceDelta(
    anchor, pillars, deltas, grid, sjd.SurfaceInterpolator.BICUBIC
)
assert bicubic.validate().ok
strikes = [forward * (0.8 + 0.04 * step) for step in range(11)]
bilinear_vols = [delta_surface.get_volatility(horizon, strike, forward) for strike in strikes]
bicubic_vols = [bicubic.get_volatility(horizon, strike, forward) for strike in strikes]
assert any(
    not math.isclose(left, right, rel_tol=1e-8, abs_tol=1e-8)
    for left, right in zip(bilinear_vols, bicubic_vols)
)

figure, axis = plt.subplots(figsize=(6, 3.5))
axis.plot(strikes, bilinear_vols, marker="o", label="bilinear")
axis.plot(strikes, bicubic_vols, marker="o", label="bicubic")
axis.set_xlabel("strike")
axis.set_ylabel("volatility")
axis.legend()
figure.tight_layout()
figure


## Where a curve is expected

A surface is a volatility source. It can be the environment's volatility, and the volatility a
forward rolls with. The forward's level is still the pillar; the surface is there for the roll
and for options.


In [ ]:
rolled = sjd.ForwardCurve(
    anchor,
    pillars,
    [100.0, 102.0, 106.0],
    roll=sjd.ForwardRoll.FORWARD_VOLATILITY,
    volatility=delta_surface,
)
environment = sjd.RatesEnvironment(anchor)
environment.set_frs(rolled)
environment.set_drs(sjd.DiscountCurve(anchor, pillars, [1.0, 0.98, 0.93]))
environment.set_vrs(delta_surface)
assert rolled.validate().ok
assert environment.validate().ok
assert math.isclose(rolled.get_forward(horizon), 106.0)
print(environment)


## What can go wrong

A grid with fewer rows than strike coordinates reports
`VOLATILITY_SURFACE_DIMENSION_MISMATCH`. Strike coordinates out of order report
`VOLATILITY_SURFACE_STRIKES_NOT_INCREASING`. A delta that cannot be solved raises
`NumericalError`. That is an `ArithmeticError`, and it is not a `ValueError`.


In [ ]:
short_row = [0.20, 0.20]
short = sjd.VolatilitySurfaceDelta(
    anchor,
    [anchor, horizon],
    [10.0, 25.0, 50.0],
    [short_row, short_row],
    sjd.SurfaceInterpolator.BILINEAR,
)
dimension_report = short.validate()
assert not dimension_report.ok
assert dimension_report.issues[0].code == "VOLATILITY_SURFACE_DIMENSION_MISMATCH"
dimension_report


In [ ]:
unordered = sjd.VolatilitySurfaceDelta(
    anchor,
    [anchor, horizon],
    [50.0, 10.0, 90.0],
    [[0.20, 0.20], [0.18, 0.18], [0.22, 0.22]],
    sjd.SurfaceInterpolator.BILINEAR,
)
order_report = unordered.validate()
assert not order_report.ok
assert order_report.issues[0].code == "VOLATILITY_SURFACE_STRIKES_NOT_INCREASING"
order_report


In [ ]:
try:
    delta_surface.strike_from_delta(horizon, -1.0, forward)
except sjd.NumericalError as error:
    assert isinstance(error, ArithmeticError)
    assert not isinstance(error, ValueError)
    assert error.code == "NUMERICAL"
    print(type(error).__name__, error)
else:
    raise AssertionError("a delta outside the smile should not solve")


## What it logs

The failed solve writes `Could not convert a delta to a strike` at `WARNING` on `sjd._core`.


In [ ]:
class Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []

    def emit(self, record):
        self.records.append(record)

captured = Collect()
logging.getLogger("sjd._core").addHandler(captured)
try:
    delta_surface.strike_from_delta(horizon, -1.0, forward)
except sjd.NumericalError:
    pass
messages = [record.getMessage() for record in captured.records]
assert any("Could not convert a delta to a strike" in message for message in messages)
assert all(record.levelno == logging.WARNING for record in captured.records)
for message in messages:
    print("WARNING", message)


## Save and reload

A surface pickles from its dates, strike grid, and volatilities. The restored surface agrees
at the same strike.


In [ ]:
import pickle
import tempfile
from pathlib import Path

scratch = tempfile.TemporaryDirectory()
path = Path(scratch.name) / "surface.pkl"
path.write_bytes(pickle.dumps(delta_surface))
restored = pickle.loads(path.read_bytes())
assert math.isclose(
    restored.get_volatility(horizon, strike_50, forward),
    delta_surface.get_volatility(horizon, strike_50, forward),
)
print("reloaded", restored)
